In [1]:
import os
os.chdir("..")
print("Working directory:", os.getcwd())

Working directory: c:\Users\Lenovo\Desktop\KYC_Project


In [2]:
from ultralytics import YOLO
from PIL import Image

doc_model = YOLO("ml/docservice/models/doc_field_detector_v3.pt")

def detect_and_crop(image_path, conf_threshold=0.5, save_crops=False, output_dir="cropped_fields"):
    results = doc_model.predict(source=image_path, conf=conf_threshold, verbose=False)
    original_image = Image.open(image_path).convert("RGB")
    detections = {}
    for box in results[0].boxes:
        cls_id = int(box.cls[0])
        cls_name = doc_model.names[cls_id]
        conf = float(box.conf[0])
        x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())
        if cls_name in detections and detections[cls_name]["confidence"] >= conf:
            continue
        crop = original_image.crop((x1, y1, x2, y2))
        detections[cls_name] = {"crop": crop, "confidence": conf, "bbox": (x1, y1, x2, y2)}
    if save_crops:
        os.makedirs(output_dir, exist_ok=True)
        base_name = os.path.splitext(os.path.basename(image_path))[0]
        for cls_name, data in detections.items():
            save_path = os.path.join(output_dir, f"{base_name}_{cls_name}.jpg")
            data["crop"].save(save_path)
    return detections

In [3]:
!pip install easyocr


[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
import easyocr
import numpy as np

reader = easyocr.Reader(['ne'], gpu=True)

def run_ocr_on_crop(pil_image, upscale=3):
    w, h = pil_image.size
    upscaled = pil_image.resize((w * upscale, h * upscale), Image.LANCZOS)
    img_array = np.array(upscaled)
    results = reader.readtext(img_array, text_threshold=0.4, low_text=0.3, link_threshold=0.3)
    extracted_text = " ".join([text for (_, text, conf) in results])
    avg_conf = sum([conf for (_, _, conf) in results]) / len(results) if results else 0
    return extracted_text, avg_conf, results

In [5]:
!pip install python-Levenshtein


[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [6]:
import Levenshtein

def normalize_text(text):
    text = text.strip()
    text = " ".join(text.split())
    return text.replace(" ", "")

def normalize_number(text):
    text = text.strip()
    allowed = set("०१२३४५६७८९0123456789")
    return "".join(ch for ch in text if ch in allowed)

def match_name_field(ocr_text, user_text, threshold=0.80):
    ocr_norm = normalize_text(ocr_text)
    user_norm = normalize_text(user_text)
    if not ocr_norm or not user_norm:
        return {"match": False, "similarity": 0.0, "reason": "empty text on one side"}
    distance = Levenshtein.distance(ocr_norm, user_norm)
    max_len = max(len(ocr_norm), len(user_norm))
    similarity = 1 - (distance / max_len)
    return {"match": similarity >= threshold, "similarity": round(similarity, 3),
            "ocr_normalized": ocr_norm, "user_normalized": user_norm}

def match_citizenship_number(ocr_text, user_text):
    ocr_digits = normalize_number(ocr_text)
    user_digits = normalize_number(user_text)
    if not ocr_digits or not user_digits:
        return {"match": False, "reason": "empty digits on one side", "ocr_digits": ocr_digits, "user_digits": user_digits}
    return {"match": ocr_digits == user_digits, "ocr_digits": ocr_digits, "user_digits": user_digits}

def match_gender(ocr_text, user_selection):
    ocr_norm = normalize_text(ocr_text)
    user_norm = normalize_text(user_selection)
    return {"match": ocr_norm == user_norm, "ocr_normalized": ocr_norm, "user_normalized": user_norm}

In [12]:
def verify_kyc_document(image_path, user_data, conf_threshold=0.5, required_fields=None):
    """
    Full pipeline: detect fields -> OCR text fields -> match against user data.
    
    required_fields: list of field names that MUST be present in user_data and MUST match
                      for overall_match to be True. Defaults to all 5 text fields.
    """
    if required_fields is None:
        required_fields = ["fname", "mname", "name", "c_no", "gender"]

    result = {"image": image_path, "fields": {}, "missing_fields": [], "overall_match": False}
    detections = detect_and_crop(image_path, conf_threshold=conf_threshold)

    text_field_matchers = {
        "fname": match_name_field, "mname": match_name_field, "name": match_name_field,
        "c_no": match_citizenship_number, "gender": match_gender,
    }
    non_text_fields = ["photo", "emblem", "logo"]
    all_match = True
    fields_checked = 0

    for field_name, matcher_fn in text_field_matchers.items():
        expected_value = user_data.get(field_name)

        if field_name not in detections:
            result["missing_fields"].append(field_name)
            result["fields"][field_name] = {"status": "not_detected"}
            if field_name in required_fields:
                all_match = False
            continue

        if expected_value is None:
            result["fields"][field_name] = {"status": "no_user_value_provided"}
            if field_name in required_fields:
                all_match = False   # <-- FIX: missing required input now fails the check
            continue

        crop = detections[field_name]["crop"]
        detection_conf = detections[field_name]["confidence"]
        ocr_text, ocr_conf, _ = run_ocr_on_crop(crop)
        match_result = matcher_fn(ocr_text, expected_value)
        result["fields"][field_name] = {
            "status": "checked", "detection_confidence": round(detection_conf, 3),
            "ocr_text": ocr_text, "ocr_confidence": round(ocr_conf, 3),
            "match": match_result["match"], "details": match_result,
        }
        fields_checked += 1
        if not match_result["match"]:
            all_match = False

    for field_name in non_text_fields:
        result["fields"][field_name] = {"status": "detected" if field_name in detections else "not_detected"}

    result["overall_match"] = all_match and fields_checked > 0
    result["fields_checked"] = fields_checked
    return result

In [13]:
test_image = os.path.join("valid/images", os.listdir("valid/images")[0])
user_data = {
    "fname": "गोघाल उप्रेती", "mname": "सविता बस्नेत", "name": "सुनिता उप्रेती",
    "c_no": "२८-०१-७८-००३३९", "gender": "महिला",
}
result = verify_kyc_document(test_image, user_data)
print("Overall match:", result["overall_match"])

Overall match: True


In [14]:
# First, get real OCR ground truth for one image so we know what "correct" looks like
test_image = os.path.join("valid/images", os.listdir("valid/images")[0])
detections = detect_and_crop(test_image)

ground_truth = {}
for field in ["fname", "mname", "name", "c_no", "gender"]:
    if field in detections:
        text, conf, _ = run_ocr_on_crop(detections[field]["crop"])
        ground_truth[field] = text

print("Ground truth OCR for this image:")
print(ground_truth)
print()

# Now build test cases: mix of correct, subtly wrong, and completely wrong data
test_cases = {
    "1. Exact correct data": {
        "fname": ground_truth["fname"],
        "mname": ground_truth["mname"],
        "name": ground_truth["name"],
        "c_no": ground_truth["c_no"],
        "gender": ground_truth["gender"],
    },
    "2. Correct but c_no has no dashes (should still match)": {
        **ground_truth,
        "c_no": ground_truth["c_no"].replace("-", "").replace(" ", ""),
    },
    "3. One digit wrong in c_no (should FAIL)": {
        **ground_truth,
        "c_no": ground_truth["c_no"][:-1] + "०",  # change last digit
    },
    "4. Wrong gender (should FAIL)": {
        **ground_truth,
        "gender": "पुरुष" if ground_truth["gender"] != "पुरुष" else "महिला",
    },
    "5. Completely wrong name (should FAIL)": {
        **ground_truth,
        "fname": "राम श्रेष्ठ",
    },
    "6. Missing c_no field entirely (should be flagged, not silently pass)": {
        "fname": ground_truth["fname"],
        "mname": ground_truth["mname"],
        "name": ground_truth["name"],
        "gender": ground_truth["gender"],
        # c_no intentionally omitted
    },
    "7. Empty string for fname (should FAIL, not crash)": {
        **ground_truth,
        "fname": "",
    },
}

for label, user_data in test_cases.items():
    result = verify_kyc_document(test_image, user_data)
    print(f"{label}")
    print(f"  Overall match: {result['overall_match']}")
    for field in ["fname", "mname", "name", "c_no", "gender"]:
        field_result = result["fields"].get(field, {})
        status = field_result.get("status")
        match = field_result.get("match")
        print(f"    {field}: status={status}, match={match}")
    print()

Ground truth OCR for this image:
{'fname': 'गोघाल उप्रेती', 'mname': 'सविता बस्नेत', 'name': 'सुनिता उप्रेती', 'c_no': '२८-०१ - ७८ -००३३९', 'gender': 'महिला'}

1. Exact correct data
  Overall match: True
    fname: status=checked, match=True
    mname: status=checked, match=True
    name: status=checked, match=True
    c_no: status=checked, match=True
    gender: status=checked, match=True

2. Correct but c_no has no dashes (should still match)
  Overall match: True
    fname: status=checked, match=True
    mname: status=checked, match=True
    name: status=checked, match=True
    c_no: status=checked, match=True
    gender: status=checked, match=True

3. One digit wrong in c_no (should FAIL)
  Overall match: False
    fname: status=checked, match=True
    mname: status=checked, match=True
    name: status=checked, match=True
    c_no: status=checked, match=False
    gender: status=checked, match=True

4. Wrong gender (should FAIL)
  Overall match: False
    fname: status=checked, matc